In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyscipopt import Model, quicksum
import random as rnd
import time

In [2]:
#paramterelere aşağıdaki şekilde random değerler verildi, arayüz tasarımı tamamlanırsa bu değerler kullanıcıdan alınacaktır.

K_crm = list(range(10))                 
K_mass = list(range(10, 20))           
K = K_crm + K_mass    
r_max = 6000 # bir kampanyanın en fazla recommend edilme sayısı
M = range(25000) #müşteri sayısı 
p_km = {(k, m): rnd.uniform(0.1, 1.0) for k in K for m in M} #eğilim skorları 
#print(p_ij)
z_k = {k: rnd.uniform(50, 200) for k in K}#i kampanyasının j müşterisine sunulmasından elde edilen kar
#print(z_ij)
c_k = [rnd.uniform(10, 200) for k in K] #i kampanyasının costu 
#print(c_k)
b_min = 0 
b_max = 10000000
start = time.time()

In [3]:
#optimzasyon fonksiyonu

def kampanya_optimizasyonu(K, K_crm, K_mass, M, p_km, z_k, c_k, b_min, b_max): #paramterelerimizi tanımlıyoruz
    model = Model("Kampanya_Optimizasyonu") #optimizasyon modeli başlatılır
    count = 0 #initialization for optimization value
# Karar değişkenleri için initialization
    x = {}
    y = {}

#her bir müşteri CRM kampanyası eşleşmesi yapılır, x içerisinde binary şekilde değer alırlar.
    for k in K_crm:
        for m in M:
            x[k, m] = model.addVar(vtype="BINARY", name=f"x_{k}_{m}")
#mass kampanyalarının müşterisi olmadığı için kampanyalar y içerisinde binary değer alırlar.
    for k in K_mass:
            y[k] = model.addVar(vtype="BINARY", name=f"y_{k}")

    # Amaç fonksiyonu maksimizasyondur, CRM ve Mass için kar hesaplanır ve aşağıdaki şekilde toplanır
    model.setObjective(
        quicksum(p_km[k, m] * z_k[k] * x[k, m] for k in K_crm for m in M) +
        quicksum(y[k] * z_k[k] * quicksum(p_km[k, m] for m in M) for k in K_mass),
        "maximize"
    )
#KISITLAR

    #Her bir CRM kampanyası en fazla r_max kadar recommend edilebilir
    for k in K_crm:
        model.addCons(
            sum(x[k, m] for m in M) <= r_max,
            name=f"crm_kampanya_limit_{k}"
        )

    # Mass kampanyaları için her mass kampanya en fazla 2 en az 1 kere recommend edilsin dedik, bu kısın değiştirilebilir
    model.addCons(quicksum(y[k] for k in K_mass) <= 2, name="en fazla 2 mass kampanya")
    model.addCons(quicksum(y[k] for k in K_mass) >= 1, name="en az 1 mass kampanya")


    #Her müşteriye en fazla 2 kampanya önerilebilir (CRM+Mass bütün olarak düşünüyoruz)
    for m in M:
        model.addCons(
            sum(x[k, m] for k in K_crm) + sum(y[k] for k in K_mass) <= 2,
            name=f"musteri_limit_{m}"
        )

    #Bütçe kısıtları (Gene CRM+Mass bütün olarak düşünülür, iki tip kampanyanın toplam bütçesinin max-min sınırlar içerisinde kalması sağlanır) )
    for m in M:
        total_cost = (
            sum(c_k[k] * x[k, m] for k in K_crm) +
            sum(c_k[k] * y[k] for k in K_mass)
        )
        model.addCons(total_cost >= b_min, name=f"butce_min_{m}")
        model.addCons(total_cost <= b_max, name=f"butce_max_{m}")

    #optimizasyon kodu çalışır
    model.optimize()

    if model.getStatus() == 'optimal':
        for k in K:
            for m in M:
                val = model.getVal(x[k, m]) if k in K_crm else model.getVal(y[k])
                if val > 0.5:
                    kampanya_tip = "CRM" if k in K_crm else "Mass"
    #Kampanya müşteri eşleşmeleri çıktısı bu satırda oluşur.
                    print(f"{kampanya_tip} Kampanya {k} -> Müşteri {m}")
        
    #Poblem infeasible ise dönecek sonuç
    else:
        print("Çözüm bulunamadı.")
        
    #bu kısım test için, bir müşteriye gerçekten en fazla iki kampanya önerilmiş mi, en az 1 tanesi Mass mi bunları kontrol ederiz
    #customer number değiştirilerek farklı müşteriler için kontrol sağlanabilir
    target_customer = 3634

    # CRM kampanyaları
    applied_crm = []
    for k in K_crm:
        if model.getVal(x[k, target_customer]) > 0.5:
            applied_crm.append(k)

    # Mass kampanyaları
    applied_mass = []
    for k in K_mass:
        if model.getVal(y[k]) > 0.5:
            applied_mass.append(k)

    print(f"Müşteri {target_customer} için uygulanan CRM kampanyaları: {applied_crm}")
    print(f"Müşteri {target_customer} için uygulanan Mass kampanyaları: {applied_mass} (tüm müşterilere uygulanmıştır)")

In [4]:
#optimizasyon fonksiyonunu çağırdığımız kısım
kampanya_optimizasyonu(K, K_crm, K_mass, M, p_km, z_k, c_k, b_min, b_max)

CRM Kampanya 0 -> Müşteri 14
CRM Kampanya 0 -> Müşteri 18
CRM Kampanya 0 -> Müşteri 262
CRM Kampanya 0 -> Müşteri 278
CRM Kampanya 0 -> Müşteri 292
CRM Kampanya 0 -> Müşteri 411
CRM Kampanya 0 -> Müşteri 468
CRM Kampanya 0 -> Müşteri 472
CRM Kampanya 0 -> Müşteri 505
CRM Kampanya 0 -> Müşteri 610
CRM Kampanya 0 -> Müşteri 629
CRM Kampanya 0 -> Müşteri 747
CRM Kampanya 0 -> Müşteri 758
CRM Kampanya 0 -> Müşteri 990
CRM Kampanya 0 -> Müşteri 1052
CRM Kampanya 0 -> Müşteri 1061
CRM Kampanya 0 -> Müşteri 1076
CRM Kampanya 0 -> Müşteri 1111
CRM Kampanya 0 -> Müşteri 1176
CRM Kampanya 0 -> Müşteri 1206
CRM Kampanya 0 -> Müşteri 1299
CRM Kampanya 0 -> Müşteri 1369
CRM Kampanya 0 -> Müşteri 1370
CRM Kampanya 0 -> Müşteri 1407
CRM Kampanya 0 -> Müşteri 1446
CRM Kampanya 0 -> Müşteri 1513
CRM Kampanya 0 -> Müşteri 1515
CRM Kampanya 0 -> Müşteri 1570
CRM Kampanya 0 -> Müşteri 1601
CRM Kampanya 0 -> Müşteri 1617
CRM Kampanya 0 -> Müşteri 1860
CRM Kampanya 0 -> Müşteri 1868
CRM Kampanya 0 -> Müşter

In [5]:
#kodun çalışma süresi burda hesaplanır
end = time.time()
print(f"Çalışma süresi: {end - start:.2f} saniye")


Çalışma süresi: 2796.76 saniye
